In [4]:
# connecting the cluster 
import ipyparallel as parallel
clients = parallel.Client() 
clients.activate()
print("Connected engine IDs:", clients.ids)

Connected engine IDs: [0, 1, 2]


In [5]:
%%px
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size() # This will be 3

# ==========================================
# STEP 1: Load inputs into process 0
# ==========================================
data_array = None
kernel_array = None
data_shape = None

if rank == 0:
    # Creating a 6x6 input data array and a 3x3 kernel for demonstration
    data_array = np.ones((6, 6)) 
    kernel_array = np.array([[1, 0, -1], 
                             [1, 0, -1], 
                             [1, 0, -1]])
    data_shape = data_array.shape

# ==========================================
# STEPS 2 & 3: Broadcast dimensions and kernel
# ==========================================
data_shape = comm.bcast(data_shape, root=0)
kernel_array = comm.bcast(kernel_array, root=0)

# ==========================================
# STEPS 4 & 5: Split and Scatter data
# ==========================================
local_data = None
if rank == 0:
    # Split rows equally: a 6x6 array split across 3 processes gives 2 rows per process
    split_data = np.array_split(data_array, size, axis=0) 
else:
    split_data = None

local_data = comm.scatter(split_data, root=0)

# ==========================================
# STEP 6: Padding the input array
# ==========================================
if rank == 0:
    local_data = np.vstack([np.zeros((1, data_shape[1])), local_data])
if rank == size - 1:
    local_data = np.vstack([local_data, np.zeros((1, data_shape[1]))])

local_data = np.pad(local_data, ((0,0), (1,1)), mode='constant')

# ==========================================
# STEP 7: Boundary Row Exchange (Send/Receive)
# ==========================================
top_row = np.zeros(data_shape[1] + 2)
bottom_row = np.zeros(data_shape[1] + 2)

if rank < size - 1:
    comm.Send(local_data[-1, :].copy(), dest=rank+1, tag=1)
if rank > 0:
    comm.Recv(top_row, source=rank-1, tag=1)
    local_data = np.vstack([top_row, local_data])

if rank > 0:
    row_to_send = 1 if rank == 0 else 1 
    comm.Send(local_data[row_to_send, :].copy(), dest=rank-1, tag=2)
if rank < size - 1:
    comm.Recv(bottom_row, source=rank+1, tag=2)
    local_data = np.vstack([local_data, bottom_row])

# ==========================================
# STEP 8: Compute Convolution Locally
# ==========================================
kernel_size = kernel_array.shape[0]
local_output_rows = local_data.shape[0] - kernel_size + 1
local_output = np.zeros((local_output_rows, data_shape[1]))

for i in range(local_output_rows):
    for j in range(data_shape[1]):
        region = local_data[i:i+kernel_size, j:j+kernel_size]
        local_output[i, j] = np.sum(region * kernel_array)

# ==========================================
# STEP 9: Gather output arrays at process 0
# ==========================================
final_output = comm.gather(local_output, root=0)

if rank == 0:
    final_matrix = np.vstack(final_output)
    print("Final Convolved Matrix:\n", final_matrix)

[stdout:0] Final Convolved Matrix:
 [[-2.  0.  0.  0.  0.  2.]
 [-3.  0.  0.  0.  0.  3.]
 [-3.  0.  0.  0.  0.  3.]
 [-3.  0.  0.  0.  0.  3.]
 [-3.  0.  0.  0.  0.  3.]
 [-2.  0.  0.  0.  0.  2.]]
